# 03 - Model Training

Two targets are trained here, and they support **very different claims**.

| | synthetic HSI | observed dissolved oxygen |
|---|---|---|
| label | generated by `aquanexus.data.synthetic` | measured, 公共用水域 monitoring record |
| rows | 7,314 (53 sections x 138 observations) | 138 (one per observation) |
| a good score means | the model recovered a formula this repo wrote | the model predicts a real river |

The first produces the impressive number. The second is the one worth
believing. Both ship - the habitat framing needs a bounded index and the
credibility needs a real measurement - see `scripts/train_models.py`.

Method notes and the full write-up: `docs/ML_METHODOLOGY.md`.

In [ ]:
import sys; sys.path.insert(0, '../src')
import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from aquanexus.config import settings
from aquanexus.data.dataset import (DO_FEATURES, DO_RANGE, build_state_vectors,
                                    build_water_quality_dataset, feature_columns)
from aquanexus.data.loader import filter_stations, load_many
from aquanexus.ml.evaluator import evaluate
from aquanexus.ml.models import ModelConfig
from aquanexus.ml.splits import flow_split, grouped_split, random_split, spatial_split
from aquanexus.ml.trainer import benchmark
from aquanexus.ml.validator import ModelValidator

pd.set_option('display.width', 200)

## 1. Sources

Simulated hydraulics from the HEC-RAS sweep, observed chemistry from the
monitoring record. Neither is complete on its own; the discharge measured
alongside each water sample is what joins them.

In [ ]:
files = sorted(glob.glob(str(settings.RAW_DIR / 'waterquality' / 'saitama_*.xlsx')))
observations = filter_stations(load_many(files), water_body=settings.RIVER_NAME_JA)
sweep = pd.read_csv(settings.PROCESSED_DIR / 'ayase_flow_sweep.csv')

print(f'observations : {len(observations)} samples, {observations.station.nunique()} stations')
print(f'flow sweep   : {sweep.river_station.nunique()} sections x '
      f'{sweep.discharge_bc.nunique()} discharges '
      f'({sweep.discharge_bc.min():.2f}-{sweep.discharge_bc.max():.1f} m3/s)')

In [ ]:
state = build_state_vectors(observations, sweep)
features = feature_columns(state)

water = build_water_quality_dataset(observations, sweep)
do_features = [f for f in DO_FEATURES if f in water.columns]

print(f'state vectors : {state.shape[0]} rows x {len(features)} features -> synthetic hsi')
print(f'water quality : {water.shape[0]} rows x {len(do_features)} features '
      f'-> observed dissolved_oxygen')

## 2. Why every split has to be grouped

All 53 rows from one observation carry **identical chemistry** - only the
hydraulics vary along the reach. Split those rows at random and near-duplicates
land on both sides of the boundary.

The two numbers below are how much the label varies *within* one observation
against how much it varies *between* observations.

In [ ]:
within = state.groupby('group').hsi.std().mean()
between = state.groupby('group').hsi.mean().std()
print(f'HSI sd within one observation (geometry varies)  : {within:.3f}')
print(f'HSI sd between observation means (chemistry)     : {between:.3f}')

for split in (grouped_split(state), spatial_split(state), flow_split(state),
              random_split(state)):
    print(split)

## 3. Synthetic HSI benchmark

Four models across four splits. `mean` is the constant-prediction floor: a
model that cannot clear it has learned nothing, and `skill` states the
improvement over that floor directly rather than leaving it implied by R2.

In [ ]:
splits = [grouped_split(state), spatial_split(state), flow_split(state),
          random_split(state)]
table, fitted = benchmark(state, features, splits)
table.round(4)

### R2 0.99 is a finding about the labels, not a modelling success

HSI is a deterministic, noiseless function of the features - written in this
repository, computed by the same code that built the dataset. No measurement
error, no unexplained ecological variance. The model is recovering a smooth
analytic function from its own inputs, which is close to the easiest regression
problem there is.

`ML_STRATEGY.md` §9.1 expected R2 0.80-0.88 and linear regression around 0.55.
Both are badly exceeded, and the reason is methodological rather than a
modelling triumph.

How easy is it? A depth-4 decision tree on **single features**:

In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.tree import DecisionTreeRegressor


def probe(columns):
    """Grouped-CV R2 of a depth-4 tree on a small feature subset."""
    X, y, groups = state[columns], state.hsi, state.group
    predictions = np.empty(len(state))
    for train_idx, test_idx in GroupKFold(n_splits=5).split(X, y, groups):
        tree = DecisionTreeRegressor(max_depth=4, random_state=42)
        tree.fit(X.iloc[train_idx], y.iloc[train_idx])
        predictions[test_idx] = tree.predict(X.iloc[test_idx])
    return evaluate(y, predictions, ' + '.join(columns), 'grouped-CV', float(y.mean()))


probes = [['depth'], ['velocity'], ['depth', 'velocity'],
          ['dissolved_oxygen'], ['water_temp']]
pd.DataFrame([probe(columns).as_row() for columns in probes])[
    ['model', 'rmse', 'r2']].round(3)

Depth alone gets most of the way there, and that is the second thing the
benchmark reveals: **the synthetic label is dominated by hydraulics, not water
quality**. 53 sections span depths of 0.1-5.6 m at a single discharge while
chemistry is held constant along the reach, so geometry varies more than
chemistry does. It is an artefact of the join - and the reason the HSI model
cannot be read as an ecological result.

In [ ]:
importance = fitted[('grouped', 'xgboost')].predictor.importances().head(8)

fig, ax = plt.subplots(figsize=(6.5, 3.2))
ax.barh(importance.index[::-1], importance.values[::-1], color='#2e86ab')
ax.set_xlabel('XGBoost feature importance (gain)')
ax.set_title('Synthetic HSI: hydraulics carry the label', loc='left',
             fontweight='bold')
plt.tight_layout()

HYDRAULIC_NAMES = {'depth', 'velocity', 'shear_stress', 'flow_area', 'top_width',
                   'froude_number', 'reynolds_number', 'wse', 'invert',
                   'energy_slope', 'discharge'}
share = importance[[f for f in importance.index if f in HYDRAULIC_NAMES]].sum()
print(f'hydraulic share of the top-8 importance: {share / importance.sum():.0%}')

## 4. The honest target: observed dissolved oxygen

One row per observation; the label is a real measurement carrying real
instrument and sampling error. Reach-averaged hydraulics at the observed
discharge are the features - which is how the hydraulic model earns its place,
since oxygen is governed partly by reaeration and reaeration depends on depth
and velocity.

BOD, nutrients and suspended solids are **deliberately excluded**: they come
from the same bottle as the target, so including them would predict one
measurement from another rather than from the river's physical state.

Cross-validation holds out **whole stations**. With 4 stations and monthly
sampling, an ungrouped fold reports how well the model recognises a station,
not how well it predicts one it has never seen.

In [ ]:
campaign = ModelValidator(water, do_features, 'dissolved_oxygen', 'station')
report = campaign.run(model_types=('linear', 'random_forest', 'xgboost'),
                      output_range=DO_RANGE)
report.table().round(3)

**Ridge beats both tree models.** With 138 rows and 10 features, gradient
boosting overfits and cross-validates worse than a regularised linear fit. The
plan assumed XGBoost would win throughout; on this dataset it does not, and
`scripts/train_models.py` ships the Ridge model because of it.

### The physics baseline fails informatively

Predict DO as saturation at the observed temperature - no fitting at all:

In [ ]:
saturation = evaluate(water.dissolved_oxygen, water.do_saturation,
                      'do saturation (physics)', 'no fit',
                      float(water.dissolved_oxygen.mean()))
print(f'RMSE {saturation.rmse:.3f} mg/L   R2 {saturation.r2:+.3f}   '
      f'Pearson {saturation.pearson:.3f}   bias {saturation.bias:+.3f} mg/L')

Right **shape**, wrong **level**: it correlates well and still scores below the
mean, because it sits about 2.4 mg/L too high everywhere. The Ayase runs a
persistent oxygen deficit against saturation - a real measured property of this
river, and exactly what a habitat model should be picking up.

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 4.2))
ax.scatter(water.dissolved_oxygen, water.do_saturation, s=22, alpha=0.75,
           color='#c0392b', edgecolor='white', lw=0.4)
limits = [water.dissolved_oxygen.min() - 0.5, water.do_saturation.max() + 0.5]
ax.plot(limits, limits, color='#1b2a41', lw=1.2, label='1:1')
ax.set_xlabel('observed DO (mg/L)')
ax.set_ylabel('saturation at observed temperature (mg/L)')
ax.set_title(f'Saturation over-predicts by {saturation.bias:+.2f} mg/L on average',
             loc='left', fontweight='bold')
ax.legend(frameon=False)
plt.tight_layout()

## 5. Does the hydraulic model earn its place? (ablation)

This is the project's central claim under test: a physics-informed
transformation of discharge should carry information the raw driver does not.
Same Ridge, same grouped CV, nested feature sets.

In [ ]:
HYDRAULICS = ['reach_depth', 'reach_velocity', 'reach_top_width', 'reach_froude']
SETS = {
    'water temperature only': ['water_temp'],
    '+ season': ['water_temp', 'month_sin', 'month_cos'],
    '+ raw discharge': ['water_temp', 'month_sin', 'month_cos', 'discharge'],
    '+ HEC-RAS hydraulics': ['water_temp', 'month_sin', 'month_cos', 'discharge',
                             *HYDRAULICS],
    'hydraulics only': HYDRAULICS,
    'water + air temperature': ['water_temp', 'air_temp'],
    'all features (shipped)': do_features,
}

rows = []
for name, columns in SETS.items():
    validator = ModelValidator(water, columns, 'dissolved_oxygen', 'station')
    predictions = validator.cross_val_predict(
        ModelConfig(model_type='linear', output_range=DO_RANGE))
    metrics = evaluate(water.dissolved_oxygen, predictions, name, 'grouped-CV',
                       float(water.dissolved_oxygen.mean()))
    rows.append({'feature set': name, 'n_features': len(columns),
                 'rmse': metrics.rmse, 'r2': metrics.r2})

ablation = pd.DataFrame(rows)
ablation['delta_r2_vs_water_temp'] = ablation.r2 - ablation.r2.iloc[0]
ablation.round(3)

Read the first four rows as a ladder. Season and raw discharge both **hurt**
(-0.016 and -0.019 R2). Replacing that same discharge with the hydraulic
model's transformation of it - depth, velocity, width, Froude number -
recovers more than it lost: +0.062 over raw discharge, +0.027 over temperature
alone. Hydraulics on their own predict nothing (R2 -0.26), because dissolved
oxygen is thermally driven first.

So the physics-informed step does carry information the raw driver does not,
which is the project's central claim demonstrated on real labels. It is a
**small** effect and should be reported as one - the row below the ladder is
there to keep it in proportion: simply adding air temperature, a free feature
requiring no hydraulic model at all, buys more (+0.073) than the entire
HEC-RAS pipeline does (+0.027).

## 6. Side by side

The comparison the project turns on.

In [ ]:
hsi_grouped = fitted[('grouped', 'xgboost')].test
best_do = report.best()

summary = pd.DataFrame([
    {'target': 'synthetic HSI', 'labels': 'GENERATED by this repo',
     'model': hsi_grouped.model, 'n': hsi_grouped.n, 'rmse': hsi_grouped.rmse,
     'r2': hsi_grouped.r2, 'what it measures': 'recovery of a known function'},
    {'target': 'dissolved oxygen', 'labels': 'observed (mg/L)',
     'model': best_do.model, 'n': best_do.n, 'rmse': best_do.rmse,
     'r2': best_do.r2, 'what it measures': 'prediction of a real river'},
])
summary.round(3)

A reader shown only the first row would conclude the project works far better
than it does. A reader shown only the second would lose the habitat framing the
project is about. `scripts/train_models.py` therefore trains and saves **both**,
each carrying its own caveats, and the API returns those caveats with every
prediction:

In [ ]:
import json

manifest = json.loads((settings.MODELS_DIR / 'manifest.json').read_text(encoding='utf-8'))
for name, meta in manifest['models'].items():
    print(f"{name}  [{meta['labels']}]  {meta['model_type']}  "
          f"n={meta['n_train']}  R2={meta['metrics']['r2']:.3f}")
    for caveat in meta['caveats']:
        print(f'   - {caveat}')
    print()

## Carried forward

Open debt that touches everything above, tracked in `DEVLOG.md`:

- The DO model beats a persistence baseline by only 0.06 R2, and loses to it on
  MAE. That comparison lives in `05_model_validation.ipynb`.
- It under-predicts oxygen by about 2 mg/L at low flow - which is exactly where
  oxygen stress matters. Same notebook.
- 4 of the 53 cross-sections cut through constrictions (RS 12500 / 14000 /
  18000 / 24000). The validator flags them; they are **not** excluded, so they
  are inside every hydraulic feature used here.
- Manning's n is uncalibrated - no gauged rating curve was available for this
  reach - so depth and velocity carry an unquantified systematic error.